In [14]:
# Please execute/shift-return this cell everytime you run the notebook.  Don't edit it. 
%load_ext autoreload
%autoreload 2
from notebook import * 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Matrix tiling algorithm with Transpositon



In [15]:
render_code("matrix_mul/blockmm_transpose_reg.c", show=["//START","//END"])

// matrix_mul/blockmm_transpose_reg.c:63-86 (24 lines)
//START
void blockmm_transpose(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  int i,j,k, ii, jj, kk;
  double result = 0;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
            {
              result = 0;
              for(kk = k; kk < k+tile_size; kk++)
                result += a[ii][kk]*b[jj][kk];
              c[ii][jj] += result;
            }
      }
    }
  }  
}
//END

In [19]:
render_code("matrix_mul/blockmm_transpose_reg.c")

// matrix_mul/blockmm_transpose_reg.c:1-86 (86 lines)
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include "perfstats.h"

void blockmm_transpose(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K);
uint64_t tile_size=1;
int ARRAY_SIZE=500;
int main(int argc, char **argv)
{
  double **a, **b, **c, **b_t;
  uint64_t i, j, k, M, K, N;
  /* initialize a, b */
  if(argc == 2)
    tile_size = atoi(argv[1]);
  if(argc >= 3)
  {
    ARRAY_SIZE = atoi(argv[1]);
    tile_size = atoi(argv[2]);
  }
  M=K=N=ARRAY_SIZE;
  a = (double **)malloc(ARRAY_SIZE*sizeof(double *));
  for(i = 0; i < ARRAY_SIZE; i++)
  {
    a[i] = (double *)malloc(ARRAY_SIZE*sizeof(double));
    for(j = 0; j < ARRAY_SIZE; j++)
      a[i][j] = i+j;
  }
  b = (double **)malloc(ARRAY_SIZE*sizeof(double *));
  for(i = 0; i < ARRAY_SIZE; i++)
  {
    b[i] = (double *)malloc(ARRAY_SIZE*sizeof(double));
    for(j = 0; j < ARRAY_SIZE; j++)
      b[i][j] = i+j;
  }

  c = (double **)malloc(ARRAY_SIZE*sizeof(double *));
  for(i = 0; i < ARRAY_SIZE; i++)
  {
    c[i] = (double *)malloc(ARRAY_SIZE*sizeof(double));
    for(j = 0; j < ARRAY_SIZE; j++)
      c[i][j] = 0;
  }
  printf("%d,%lu,",ARRAY_SIZE,tile_size);
  perfstats_init();
  perfstats_enable();
  b_t = (double **)malloc(ARRAY_SIZE*sizeof(double *));
  //TRAN_START
  for(i = 0; i < ARRAY_SIZE; i++)
  {
    b_t[i] = (double *)malloc(ARRAY_SIZE*sizeof(double));
    for(j = 0; j < ARRAY_SIZE; j++)
      b_t[i][j] = b[j][i];
  }
  //TRAN_END
  blockmm_transpose(a, b_t, c, M, N, K);
  perfstats_disable();
  perfstats_print();
  perfstats_deinit();
  fprintf(stderr,"%lf,",c[rand()%ARRAY_SIZE][rand()%ARRAY_SIZE]);
  return 0;
}
//START
void blockmm_transpose(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  int i,j,k, ii, jj, kk;
  double result = 0;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
            {
              result = 0;
              for(kk = k; kk < k+tile_size; kk++)
                result += a[ii][kk]*b[jj][kk];
              c[ii][jj] += result;
            }
      }
    }
  }  
}
//END

In [17]:
! cd matrix_mul; rm blockmm_transpose_reg; make blockmm_transpose_reg; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_transpose_reg.csv 
for tile_size in (4, 8, 16, 32, 64, 128, 256, 512):
    ! ./matrix_mul/blockmm_transpose_reg 2048 {tile_size} >> ./matrix_mul/blockmm_transpose_reg.csv 

gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm_transpose_reg.c perfstats.c -o blockmm_transpose_reg
10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,

In [20]:
display_df_mono(render_csv("matrix_mul/blockmm_transpose_reg.csv"))

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,4,88420275497,19260798162,0.217832,0.179103,3.449672,0.012464,323402802,25947195171
1,1,2048,8,60794386700,16316765393,0.268393,0.179153,2.923191,0.006240,99179001,15893700615
2,2,2048,16,49275504187,10252678197,0.208068,0.179006,1.835293,0.062430,750102701,12015109307
3,3,2048,32,43909426178,9882611247,0.225068,0.179029,1.769276,0.101084,1038323461,10271847097
4,4,2048,64,41302516469,10209904883,0.247198,0.179073,1.828317,0.082142,775254954,9437961457
5,5,2048,128,40017565663,11807556573,0.295059,0.179148,2.115303,0.061286,553401332,9029855550
6,6,2048,256,39381375103,14038809390,0.356483,0.179092,2.514241,0.020085,177324389,8828627936
7,7,2048,512,39066967651,16255933938,0.416104,0.179067,2.910896,0.013996,122176548,8729410035


In [ ]:
render_code("matrix_mul/rect_blockmm_trans.c", show=["//START","//END"])

In [ ]:
! cd matrix_mul; make rect_blockmm_trans; echo "size,tile_size_x,tile_size_y,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 8 8 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 8 16 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 16 8 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 16 16 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 32 8 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 32 16 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 64 8 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 128 8 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 256 8 >> ./matrix_mul/rect_blockmm_trans.csv
display_df_mono(render_csv("matrix_mul/rect_blockmm_trans.csv"))

## Prefetch

x86 provide prefetch instructions. As a programmer, you may insert ```_mm_prefetch``` in x86 programs to perform software prefetch for your code. The gcc compiler also has a flag ```-fprefetch-loop-arrays``` to automatically insert software prefetch instructions.

### Using prefetch in matrix transpose code

The following example is a highly optimized matrix transpose code. In the example, we try to prefetch the next row.

In [9]:
render_code("./prefetch/transpose.cpp", lang="c++", show=["//START", "//END"])

// ./prefetch/transpose.cpp:42-63 (22 lines)
    //START
    //  Iterate each row.
    f_vector *y_iter = T;
    do{
        //  Iterate each column.
        f_vector *ptr_x = y_iter + block;
        f_vector *ptr_y = y_iter + row_size;

        do{

#ifdef ENABLE_PREFETCH
            _mm_prefetch((char*)(ptr_y + row_size),_MM_HINT_T0);
#endif
            swap_block(ptr_x,ptr_y,block);

            ptr_x += block;
            ptr_y += row_size;
        }while (ptr_y < stop_T);

        y_iter += iter_size;
    }while (y_iter < end);
    //END

Now, let's take a look of what's happening!

In [ ]:
! cd prefetch; make clean; make
# ! echo "Without prefetch -- the baseline"; ssh htseng@celebi "lscpu | grep Model; cd courses/CS203/demo/memory/prefetch/; ./transpose"
! echo "Without prefetch -- the baseline"
! lscpu | grep Model
! ./prefetch/transpose
! echo "With prefetch"
! ./prefetch/transpose_prefetch

Let's try a different machine now.

In [ ]:
! ssh htseng@xerneas "cd /nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@xerneas  "/nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@xerneas  "/nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/prefetch/transpose_prefetch"

In [ ]:
! ssh htseng@blissey "cd /nfshome/htseng/courses/CSE142/demo/memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@blissey  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@blissey  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose_prefetch"

In [ ]:
! ssh htseng@eevee "cd /nfshome/htseng/courses/CSE142/demo/memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@eevee  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@eevee  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose_prefetch"


-- It doesn't work always!

In [ ]:
render_code("matrix_mul/blockmm_interchange.c", show=["//START","//END"])

In [ ]:
! cd matrix_mul; rm -f blockmm_interchange; make blockmm_interchange; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_interchange.csv
! ./matrix_mul/blockmm_interchange 2048 8 >> ./matrix_mul/blockmm_interchange.csv 
! ./matrix_mul/blockmm_interchange 2048 16 >> ./matrix_mul/blockmm_interchange.csv 
! ./matrix_mul/blockmm_interchange 2048 32 >> ./matrix_mul/blockmm_interchange.csv 
! ./matrix_mul/blockmm_interchange 2048 64 >> ./matrix_mul/blockmm_interchange.csv
! ./matrix_mul/blockmm_interchange 2048 128 >> ./matrix_mul/blockmm_interchange.csv
! ./matrix_mul/blockmm_interchange 2048 256 >> ./matrix_mul/blockmm_interchange.csv
! cd matrix_mul; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm.csv
! ./matrix_mul/blockmm 2048 16 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 32 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 64 >> ./matrix_mul/blockmm.csv
! ./matrix_mul/blockmm 2048 128 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 256 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 512 >> ./matrix_mul/blockmm.csv 
display_df_mono(render_csv("matrix_mul/blockmm.csv"))
display_df_mono(render_csv("matrix_mul/blockmm_interchange.csv"))
